#### Abstract

We work with two types of flight data: accident traces and non-accident traces. Accident traces are from aircraft involved in accidents, while non-accident traces are randomly sampled from general air traffic. Together, these allow us to compare flight behavior between accident and non-accident flights.

ADS-B (Automatic Dependent Surveillance–Broadcast) signals are automatically transmitted by aircraft during flight and contain position, altitude, speed, and identification data. The dataset is collected by a community-driven network of volunteers who operate private ADS-B receivers around the world and make the data openly accessible.

To build this dataset, we used two sources:

1. **Aviation Safety Network (ASN):** We used the ASN Wikibase database to identify all recorded aviation accidents since 2023. From each entry we extracted the crash date, time, and the aircraft's identifier. ASN reports times in local time unless otherwise specified.

2. **ADSB.lol:** Using the hex codes and crash dates from ASN, we downloaded the corresponding ADS-B traces for each accident aircraft, covering a window of ±1 day around the crash date to ensure the complete accident flight is captured. 
Additionally, we downloaded traces from randomly selected dates as a baseline for comparison. ADSB.lol times are in UTC. 

Useful Reading for better understanding: https://tech.marksblogg.com/global-flight-tracking-adsb.html

#### Import Librariers and Data

In [ ]:
import tarfile
import json
import gzip
import pandas as pd
import matplotlib.pyplot as plt
# import matplotlib.colors as colors
%matplotlib inline



from paths import *
from src.common.data_integrity import *
#from src.common.statistical_plots import * 
from src.common.data_io import *
from src.traces.parquet_io import *



#### ADBS.lol TAR Archive structure


```text
TAR archive
├── traces/
│   └── <subfolder>/
│       └── <trace>.json        (gzip-compressed)
│           └── JSON trace data
│
├── acas/
│   ├── acas.json.gz            (gzip-compressed NDJSON)
│   └── acas.csv.gz             (gzip-compressed text / log format) 
│
└── heatmap/
    └── *.bin.ttf              (TrueType font files used for heatmap visualization) 


##### Import Traces data from ADBS.lol Datasets (non-accident)

Let's print out our files

In [3]:
files_no_crash = [f.name for f in TRACES_NO_ACCIDENT.iterdir() if f.is_file() and f.suffix == ".tar"]

for f in files_no_crash:
    print(f)

v2023.08.30-planes-readsb-prod-0.tar
v2023.08.31-planes-readsb-prod-0.tar
v2023.09.01-planes-readsb-prod-0.tar
v2023.10.03-planes-readsb-prod-0.tar
v2023.10.04-planes-readsb-prod-0.tar
v2023.10.05-planes-readsb-prod-0.tar
v2024.01.05-planes-readsb-prod-0.tar
v2024.01.06-planes-readsb-prod-0.tar
v2024.01.07-planes-readsb-prod-0.tar
v2024.12.28-planes-readsb-prod-0.tar
v2024.12.29-planes-readsb-prod-0.tar
v2025.06.11-planes-readsb-prod-0.tar
v2025.12.25-planes-readsb-prod-0.tar


Before we will create parquet files from our very big datasets from ADSB.lol traces, let's first examine how our raw traces datasets look like in structure and types

In [4]:
path = TRACES_NO_ACCIDENT / "v2023.08.31-planes-readsb-prod-0/traces/1a/trace_full_00a61a.json"

with gzip.open(path, "rt", encoding="utf-8") as f:
    data = json.load(f)

In [5]:
for k, v in data.items():
    print(f"{k}: {type(v)}")

icao: <class 'str'>
r: <class 'str'>
t: <class 'str'>
dbFlags: <class 'int'>
desc: <class 'str'>
timestamp: <class 'float'>
trace: <class 'list'>


### Traces Dataset Columns

- **icao** (`str`)  
  ICAO 24-bit aircraft identifier.

- **r** (`str`)  
  Aircraft registration / tail number.

- **t** (`str`)  
  Aircraft type code.

- **dbFlags** (`int`)  
  Integer database flag field.

- **desc** (`str`)  
  Human-readable aircraft description.

- **year** (`str`)  
  Aircraft year (probably Year of Construction), stored as a string. 
  (Not available in this example)

- **timestamp** (`float`)  
  Base Unix timestamp of the trace in seconds.

- **trace** (`list`)  
  List of positional trace entries stored as compact arrays.

In [7]:
print(len(data["trace"]))

231


Let’s look at one of the traces. Each trace is a time-ordered sequence of ADS-B messages representing the state of an aircraft (position, speed, altitude, and heading) at successive points during its flight.

In [6]:
print(len(data["trace"][1]))

14


In [8]:
data["trace"][0]

[16303.82,
 -25.920798,
 27.950592,
 5000,
 161.2,
 55.6,
 5,
 1984,
 {'type': 'adsb_icao',
  'flight': 'ZSPKN   ',
  'alt_geom': 5475,
  'track': 55.62,
  'geom_rate': 1984,
  'squawk': '4007',
  'emergency': 'none',
  'category': 'A2',
  'nic': 8,
  'rc': 186,
  'version': 0,
  'nac_p': 8,
  'nac_v': 2,
  'sil': 2,
  'sil_type': 'perhour',
  'alert': 0,
  'spi': 0},
 'adsb_icao',
 5475,
 1984,
 None,
 None]

### Trace Normalization Strategy

Trace entries are stored as positional arrays with mixed and optional fields.  
The structure is not deterministic: metadata may appear in some entries, while
others contain `None` values at the same positions. Therefore, applying a fixed
schema directly to raw trace arrays is not reliable.

To handle this, we first **normalize each trace entry into a structured record**
before applying any schema or writing to Parquet.

The normalization step:

- converts positional arrays into named fields
- handles optional metadata dictionaries
- keeps missing values as `None`
- extracts optional aircraft metadata when present
- preserves all available information in a consistent format

Workflow:

1. Load raw JSON trace file  
2. Iterate over `trace` entries  
3. Convert each entry into a normalized dictionary  
4. Write normalized records to JSONL  
5. Load JSONL into DuckDB and infer schema automatically  
6. Export to Parquet  

This approach avoids assumptions about trace structure and produces a stable,
schema-friendly dataset for downstream processing.

This approach is based on the workflow from https://tech.marksblogg.com/global-flight-tracking-adsb.html 

#### Creating Parquet Files

In [9]:
json_files = glob(str(TRACES_NO_ACCIDENT / '*/traces/*/*.json'))
with open("json_files.txt", "w", encoding="utf-8") as f:
    for p in json_files:
        f.write(p + "\n")

In [10]:
len(json_files)

64403

In total we have around 65 thousand JSON files, each representing a flght. As each of these flights have very big trace data, we decdided to work with 65 thousand flights as for now, we still have more flights, which we will use if necessary.

In [ ]:
convertToJsonL(json_files, PARQUET_TRACES_NO_CRASH, gzipCompressed = True)

After converting our compressed traces JSON files to enriched JSONL files, we will convert each JSONL file into a DuckDB table and then into its own Parquet file. This step will not be done on Python. 

You can read about the process in https://tech.marksblogg.com/global-flight-tracking-adsb.html (see section/caption: "Creating Enriched Parquet Files")

The enriched JSONL files had around 120 GB.
After concerting these files to Parquet, we are left with ~ 3 GB of Parquet files.

#### Import Traces data from ADBS.lol Datasets (accident)

For the accident dataset we downloaded traces by hex-code and accident-day. On top of that, we also downloaded the traces from the previous and next days for these hex-codes. We will later clean all traces which don't belong to the accident-traces.

Compared to the non-accident flight traces, the accident datasets are significantly smaller in size. This is due to the fact that accident-related data was collected very specifically and does not cover large continuous time ranges like the regular ADS-B traces.

Despite the smaller volume, it is important to ensure a consistent data structure across all datasets. Therefore, similar to the non-accident data pipeline, the accident JSON files are also processed using JSONL, DuckDB and converted then into Parquet format.

This approach provides several advantages:
- A unified data format across all datasets
- Improved performance for downstream processing

By standardizing both accident and non-accident data into Parquet, we ensure a clean and scalable foundation for further analysis and modeling.

In [8]:
json_files = glob(str(TRACES_CRASH / '*.json'))

In [9]:
len(json_files)

527

In [ ]:
convertToJsonL(json_files, PARQUET_TRACES_CRASH, gzipCompressed = True)

We downloaded traces for each accident aircraft from ADSB.lol using the hex code, covering the accident date as well as one day before and one day after (±1 day). The extra days serve as a buffer to ensure the complete accident flight is captured, even if it started late the previous day and crossed the UTC day boundary. ASN reports times in local time unless otherwise specified. The ASN accident times are the reason that we simply couldn't cut at the accident day and had to inculde the next days also.

However, since many aircraft might have survived their accidents, the traces from the day after the accident may contain normal flights unrelated to the accident. To isolate only the relevant data, we need to remove all traces recorded after the accident time. For this, we import our ASN crashes dataset and use the accident timestamps to filter accordingly.

Similarly, the traces from the day before the accident might contain regular flights that are unrelated to the accident. Commercial aircraft might operate daily and also multiple times within a day, so it is expected to find multiple flights per hex code across our ±1 day window for the same aircraft. We keep only the flights leading up to and including the accident.


For the filtering and cleansing of our accident tarces dataset, we will import our asn-accident csv and do the cleansing.

In [13]:
df = import_df(ASN_DIR, "asn_accidents.csv")

Note that only entries with a hex code are useful for matching against the ADSB.lol traces, so dropping rows without a hex code does not result in any data loss.

In [14]:
df = df[df['icao'].notna()].copy()

In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 293 entries, 0 to 620
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  293 non-null    str    
 1   time                  263 non-null    str    
 2   type                  293 non-null    str    
 3   owner/operator        292 non-null    str    
 4   icao                  293 non-null    str    
 5   registration          293 non-null    str    
 6   msn                   293 non-null    str    
 7   year of manufacture   272 non-null    float64
 8   engine model          235 non-null    str    
 9   fatalities            293 non-null    str    
 10  other fatalities      293 non-null    int64  
 11  aircraft damage       190 non-null    str    
 12  category              289 non-null    str    
 13  location_country      293 non-null    str    
 14  location              293 non-null    str    
 15  phase                 293 non-null    s

In [ ]:
df.to_csv('asn_accidents_v1.csv', index=False)